# RAG查询转换与评估演示程序

本notebook演示了RAG系统中的核心组件：
1. **文档相关性评估** - 使用LLM评估检索文档与查询的相关性
2. **查询分解** - 将复杂问题分解为多个子问题
3. **RAG处理链** - 构建完整的检索-生成流程

## 系统要求
- LangChain框架
- OpenAI API访问
- 适当的向量数据库和检索器

## 第一部分：文档相关性评估组件

### 功能描述
构建一个评估链，用于判断检索到的文档与用户查询的相关性。

In [ ]:
# 导入必要的库
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

print("✅ 基础库导入完成")

In [ ]:
# 定义数据模型
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

print("✅ 数据模型定义完成")
print(f"📋 模型字段: {list(GradeDocuments.__fields__.keys())}")

In [ ]:
# 初始化LLM模型用于相关性评估
# TODO：改为我们使用的模型
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
     model="qwen-plus",
    api_key="sk-e4b7b6386950428bb71c658d47da47ef",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

print("✅ 相关性评估LLM初始化完成")
print(f"🤖 模型类型: {type(structured_llm_grader)}")

In [ ]:
# 构建相关性评估提示模板
system = """You are a grader assessing relevance of a retrieved document to a user question. 
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant. 
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
])

# 创建相关性评估链
retrieval_grader = grade_prompt | structured_llm_grader

print("✅ 相关性评估链构建完成")
print("🔗 链结构: 提示模板 -> 结构化输出模型")

## 第二部分：查询分解组件

### 功能描述
将复杂用户问题分解为多个可独立回答的子问题，提高检索效率。

In [ ]:
# 导入查询分解相关库
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

print("✅ 查询分解库导入完成")

In [ ]:
# 定义查询分解提示模板
template4decomposition = """You are a helpful assistant that generates multiple sub-questions related to an input question. 
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. 
Generate multiple search queries related to: {question} 
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template4decomposition)

print("✅ 查询分解提示模板创建完成")
print("📝 模板指导: 将复杂问题分解为3个独立子问题")

In [ ]:
# 初始化分解用LLM模型
# TODO：改为我们使用的模型，实际使用中和评估模型一致即可
#llm = ChatOpenAI(temperature=0)

print("✅ 分解用LLM模型初始化完成")
print("🤖 模型温度设置为0以获得确定性输出")

In [ ]:
# 创建查询分解处理链
generate_queries_decomposition = (
    prompt_decomposition 
    | llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

print("✅ 查询分解处理链构建完成")
print("🔗 链结构: 提示模板 -> LLM -> 字符串解析 -> 列表分割")

In [ ]:
# 测试查询分解功能
# TODO：改为用户的实际题问
question = "基于大语言模型（LLM）的自主智能体系统的主要组件有哪些？"
#question = "What are the main components of an LLM-powered autonomous agent system?"
print(f"🔍 原始问题: {question}")
print("⏳ 正在分解查询...")

# 注意：需要实际的retriever才能执行
try:
    questions = generate_queries_decomposition.invoke({"question": question})
    print(f"✅ 查询分解完成，生成了 {len(questions)} 个子问题")
    for i, q in enumerate(questions, 1):
        print(f"  {i}. {q}")
except Exception as e:
    print(f"⚠️  需要配置retriever才能执行: {e}")
    questions = []  # 占位符

## 第三部分：文档相关性评估流程

### 功能描述
对检索得到的文档进行相关性评估，判断其与子问题的匹配程度。

In [ ]:
# 执行文档相关性评估
print("📊 开始文档相关性评估流程")
count=0
if questions:  # 确保有分解的子问题
    for i, q in enumerate(questions, 1):
        print(f"\n🔍 评估子问题 {i}: {q}")
        count+=1
        try:
            # TODO：获取我们想要的documents，这里需要修改
            # docs = retriever.get_relevant_documents(q)  # 需要实际的retriever
            

            #这里是只模拟第二次的检索文档
            if(count==3):
                mock_doc_txt2= """在LLM驱动的自主智能体中,规划与决策模块是实现目标拆解和行动选择的核心。其运作逻辑通常分为两层:

                首先是任务规划,LLM基于用户目标生成分步执行策略。例如,当目标是“完成市场分析报告”时,模块会拆解为“收集行业数据→分析竞品动态→提炼关键结论”等子步骤,并通过逻辑推理确保步骤连贯性。此过程中,LLM会调用自身的语义理解能力,结合上下文调整规划,若遇突发情况(如数据缺失),还能动态修正步骤。  

                其次是决策执行,模块根据规划结果选择具体行动。LLM会评估不同选项的可行性(如调用工具、生成文本或交互反馈），基于预设规则或历史经验输出最优解。例如，在信息不足时，决策模块会触发“调用搜索引擎补充数据”的行动，而非直接生成结论。  

                整个过程中,LLM通过 Prompt 工程（如思维链提示）模拟人类推理路径，同时结合外部工具接口（如计算器、数据库）增强决策的准确性和执行力，实现从抽象目标到具体行动的落地。"""
                grade = retrieval_grader.invoke({"question": q, "document": mock_doc_txt2})
            else:
                # 模拟文档内容（实际使用时替换为真实检索结果）
                mock_doc_txt1 = "这是模拟的文档内容，用于演示相关性评估过程"
                # TODO：评估documents是否与问题相关
                grade = retrieval_grader.invoke({"question": q, "document": mock_doc_txt1})
            
            # TODO：判断该文档与问题相关，后面修改逻辑
            if grade.binary_score == "yes":
                print(f"  ✅ 该文档与问题相关 (相关性评分: {grade.binary_score})")
            else:
                print(f"  ❌ 该文档与问题不相关 (相关性评分: {grade.binary_score})")
                
        
        except Exception as e:
            print(f"  ⚠️  评估过程出错: {e}")
            print("  💡 需要配置实际的retriever和文档数据")
else:
    print("⚠️  没有子问题可评估，请先执行查询分解")

## 第四部分：RAG回答生成组件

### 功能描述
基于分解的子问题和相关性评估结果，构建完整的RAG处理链。

In [ ]:
# 导入RAG处理所需库
from operator import itemgetter

print("✅ RAG处理库导入完成")

问答生成提示词模板

In [ ]:
# 定义回答生成提示模板
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

print("✅ RAG回答生成提示模板创建完成")
print("📝 模板包含: 当前问题、历史问答对、相关上下文")

In [ ]:
# 定义格式化问答对的函数
def format_qa_pair(question, answer):
    """格式化问答对为字符串形式"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

print("✅ 问答对格式化函数定义完成")
print("🔧 功能: 将问答对格式化为结构化字符串")

In [ ]:
# 初始化RAG用LLM模型
# TODO：改为我们使用的模型，实际使用中和评估模型一致即可
#llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

print("✅ RAG回答生成LLM模型初始化完成")
print("🤖 使用GPT-3.5-turbo生成最终回答")

In [ ]:
# 初始化问答对存储
q_a_pairs = ""

print("✅ 问答对存储初始化完成")
print("📚 用于存储历史问答，支持上下文关联")

In [ ]:
# 构建完整的RAG处理链
print("🔗 构建RAG处理链...")

# 注意：这里需要实际的retriever
try:
    # rag_chain = (
    #     {
    #         "context": itemgetter("question") | retriever,  # TODO：修改在这一步，实际已经获取了documents。后期要做调整。
    #         "question": itemgetter("question"),
    #         "q_a_pairs": itemgetter("q_a_pairs")
    #     } 
    #     | decomposition_prompt
    #     | llm
    #     | StrOutputParser()
    # )


    # Prompt
    prompt = """Here is a set of Q+A pairs:

Question: 大语言模型（LLM）在自主智能体系统中的核心作用是什么？
Answer: 大语言模型是自主智能体系统的核心认知引擎，主要作用体现在以下方面：
自然语言理解与生成：作为智能体与人类、多模态模块（视觉、语音等）的 “语义桥梁”，能解析复杂指令、理解多模态信息的语义，并生成自然连贯的回应，实现高效人机交互与跨模态信息整合。
上下文推理与决策支持：依托海量预训练知识和上下文学习能力，进行逻辑推理、因果分析、意图识别，为智能体的规划、决策提供深层语义和逻辑支撑，例如在复杂任务中推导步骤间的关联。
知识整合与迁移：将预训练阶段习得的通用知识（如常识、领域通识）与智能体的专属领域知识（如特定任务规则、环境信息）融合，实现知识的迁移与适配，快速适应新场景。
多任务协同的 “大脑中枢”：协调智能体的记忆、规划、行动等模块，通过自然语言的形式传递指令、整合各模块输出，推动多任务的连贯执行。

Question: 自主智能体系统中常用的记忆模块和知识管理机制有哪些？
Answer: 记忆模块
短期记忆（工作记忆）：
循环缓冲区：按时间顺序存储近期的交互信息、推理步骤、环境状态等，确保智能体对 “当前上下文” 的感知。
注意力机制（类 Transformer）：在处理信息时聚焦关键元素，模拟人类对重要信息的短期聚焦记忆。
长期记忆：
外部知识库：如向量数据库（以 Embedding 形式存储实体、关系、文档，支持高效语义检索）、知识图谱（结构化存储领域知识，明确实体间关联）。
情节记忆模块：记录智能体经历的特定事件、经验（如 “某次成功的任务执行路径”），用于后续经验复用与类比推理。
知识管理机制
知识检索与更新：
基于相似度的检索（如向量相似度匹配），从长期记忆中快速召回相关知识；
知识图谱的动态更新，支持新增实体、关系的插入，或过时知识的标记。
知识融合与推理：
利用 LLM 的推理能力，对多源知识（如通用常识 + 领域规则）进行融合，推导新的知识结论；
结合逻辑规则（如 IF-THEN）或概率图模型，实现知识的演绎、归纳推理。
知识遗忘与压缩：
基于使用频率、时效性的策略，移除冗余、过时的知识，降低记忆存储成本；
对高频使用的知识进行 “压缩式编码”（如提炼关键特征），平衡记忆效率与检索精度。

Question: 基于 LLM 的智能体如何实现规划、决策与行动执行功能？
Answer: 规划功能：
任务分解与子目标生成：LLM 通过 “链式思维（Chain of Thought）” 或 “树状规划”，将复杂任务拆解为可执行的子目标序列。例如，“举办一场线上会议” 可被分解为 “确定时间→邀请参会人→制定议程→发送通知” 等子目标。
资源与步骤的时序规划：结合环境约束（如资源限制、时间窗口），为子目标安排执行顺序，生成 “步骤 - 资源 - 时间” 的三维规划路径。
决策功能：
多选项评估与推理：LLM 针对规划阶段的候选方案，结合知识与环境状态，分析各方案的收益、风险、可行性（如 “选择方案 A 的成功率为 70%，但耗时更久；方案 B 效率高，但依赖外部 API 稳定性”）。
不确定性下的决策：利用概率推理或强化学习思维（如模拟不同决策的后续奖励），在信息不全时选择最优策略，例如在博弈场景中预测对手行为并决策。
行动执行功能：
可执行指令生成：将决策结果转化为具体的可执行指令，如控制机器人的动作序列（“向前移动 5 米→抓取物体”）、调用外部工具的 API 参数（“调用天气 API，查询北京明天的温度”）、或自然语言的操作指引（“给用户发送邮件，主题为‘会议提醒’”）。
反馈循环与动态调整：行动执行后，LLM 接收环境反馈（如工具返回结果、用户回应），分析反馈信息的语义，判断行动是否达成预期，若未达成则调整后续规划与决策（如 “邮件发送失败，重新选择发送渠道并优化内容”）。

Use these to synthesize an answer to the question: 基于大语言模型（LLM）的自主智能体系统的主要组件有哪些？
"""

    
    answer = llm.invoke(prompt)
    
    parser = StrOutputParser()
    #使用parser处理model返回的结果
    response = parser.invoke(answer)
    # 打印解析后的结果
    print("🔍 解析后的回答:", response)

    print("✅ RAG处理链构建完成")
    print("🔗 链结构: 问题输入 -> 检索上下文 -> 提示模板 -> LLM生成 -> 字符串解析")
    
except Exception as e:
    print(f"⚠️  需要配置retriever: {e}")
    print("💡 请先配置向量数据库和retriever组件")
    rag_chain = None





In [ ]:
#test
prompt = "基于大语言模型（LLM）的自主智能体系统的主要组件有哪些？"

answer = llm.invoke(prompt)
    
parser = StrOutputParser()
#使用parser处理model返回的结果
response = parser.invoke(answer)
# 打印解析后的结果
print("🔍 解析后的回答:", response)